# Classification de Tumeurs Mammaires (Bénignes / Malignes)

**Dataset :** Breast Cancer Wisconsin (Diagnostic) — [UCI ML Repository](https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic))  
**Modèles :** SVM · Random Forest · MLP

---

## 1. Chargement des Données & Analyse Exploratoire (EDA)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer

# Reproducibility
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load dataset
cancer = load_breast_cancer()

df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target           # 0 = malignant, 1 = benign
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print(f'Dataset shape : {df.shape}')
print(f'Features      : {cancer.feature_names.tolist()}')
df.head()

In [ ]:
# Class distribution
counts = df['diagnosis'].value_counts()
print('Class distribution:')
print(counts)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(counts.index, counts.values, color=['#e74c3c', '#2ecc71'], edgecolor='black', width=0.5)
ax.set_title('Distribution des classes', fontsize=14, fontweight='bold')
ax.set_xlabel('Diagnostic')
ax.set_ylabel('Nombre de cas')
for i, v in enumerate(counts.values):
    ax.text(i, v + 4, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Descriptive statistics
df.drop(columns=['target', 'diagnosis']).describe().T

In [ ]:
# Check for missing values
missing = df.isnull().sum().sum()
print(f'Missing values: {missing}  — Dataset is complete.')

In [ ]:
# Distribution of first 10 features by class
top_features = cancer.feature_names[:10].tolist()

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    for label, color in zip(['Benign', 'Malignant'], ['#2ecc71', '#e74c3c']):
        axes[i].hist(df[df['diagnosis'] == label][feat], bins=25,
                     alpha=0.6, label=label, color=color, edgecolor='white')
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)

fig.suptitle('Distribution des 10 premières caractéristiques par classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (mean features only)
mean_cols = [c for c in df.columns if 'mean' in c]
corr = df[mean_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Matrice de corrélation — caractéristiques moyennes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot of top 5 most discriminative features
key_feats = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean concavity']
pair_df = df[key_feats + ['diagnosis']]

sns.pairplot(pair_df, hue='diagnosis', palette={'Benign': '#2ecc71', 'Malignant': '#e74c3c'},
             plot_kws={'alpha': 0.5, 's': 20})
plt.suptitle('Pairplot — caractéristiques clés', y=1.01, fontsize=13, fontweight='bold')
plt.show()

---
## 2. Prétraitement des Données

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Features & target
X = cancer.data
y = cancer.target   # 0 = malignant, 1 = benign

# Train / Test split  (80 / 20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {X_train.shape[0]} samples')
print(f'Test  size : {X_test.shape[0]} samples')
print(f'Train class ratio — Benign: {y_train.mean():.2%}  Malignant: {1 - y_train.mean():.2%}')

In [ ]:
# Standard scaling (fit on train only → transform both sets)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Scaled train mean  ≈', X_train_sc.mean(axis=0)[:5].round(4))
print('Scaled train std   ≈', X_train_sc.std(axis=0)[:5].round(4))

In [ ]:
# Visualise effect of scaling on first feature
feat_idx = 0
feat_name = cancer.feature_names[feat_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in zip(
    axes,
    [X_train[:, feat_idx], X_train_sc[:, feat_idx]],
    [f'{feat_name} — avant normalisation', f'{feat_name} — après normalisation']
):
    ax.hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Fréquence')

plt.tight_layout()
plt.show()

---
## 3. Modèle SVM (Support Vector Machine)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

In [ ]:
# Hyperparameter tuning via Grid Search + 5-fold CV
param_grid_svm = {
    'C'     : [0.1, 1, 10, 100],
    'kernel': ['rbf', 'linear'],
    'gamma' : ['scale', 'auto']
}

gs_svm = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid_svm,
    cv=5, scoring='accuracy', n_jobs=-1, verbose=0
)
gs_svm.fit(X_train_sc, y_train)

print(f'Best SVM params   : {gs_svm.best_params_}')
print(f'Best CV accuracy  : {gs_svm.best_score_:.4f}')

In [ ]:
# Evaluate best SVM on test set
svm_best = gs_svm.best_estimator_
y_pred_svm  = svm_best.predict(X_test_sc)
y_prob_svm  = svm_best.predict_proba(X_test_sc)[:, 1]

acc_svm = accuracy_score(y_test, y_pred_svm)
auc_svm = roc_auc_score(y_test, y_prob_svm)

print(f'SVM — Test Accuracy : {acc_svm:.4f}')
print(f'SVM — ROC-AUC       : {auc_svm:.4f}')
print()
print(classification_report(y_test, y_pred_svm, target_names=cancer.target_names))

In [ ]:
# Confusion matrix
cm_svm = confusion_matrix(y_test, y_pred_svm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Confusion matrix ---
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues',
            xticklabels=cancer.target_names,
            yticklabels=cancer.target_names, ax=axes[0])
axes[0].set_title('SVM — Matrice de confusion', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Prédiction')
axes[0].set_ylabel('Réalité')

# --- ROC curve ---
fpr, tpr, _ = roc_curve(y_test, y_prob_svm)
axes[1].plot(fpr, tpr, color='royalblue', lw=2, label=f'SVM (AUC = {auc_svm:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_title('SVM — Courbe ROC', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Taux de faux positifs')
axes[1].set_ylabel('Taux de vrais positifs')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()